In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import yaml

shot_number = 171348
yaml_path = "../src/fusionaihub/datasets/prepare/config/default.yaml"
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

from fusionaihub.datasets.prepare.data_extraction import (
    extract_signal, 
    extract_running_time, 
    align_signal,
)
from fusionaihub.datasets.prepare.sample_processing import (
    split_samples,
    remove_empty_samples,
    save_sample,
)
from fusionaihub.datasets.prepare.signal_processing import (
    identity_transform,
    stft_transform,
    resample_transform,
)

In [3]:
start_time, end_time = extract_running_time(
    shot_number=shot_number,
    directory=Path(cfg["raw_data_dir"]),
    ip_threshold=cfg["ip_threshold"],
    start_time=cfg["start_time"],
    end_time=cfg["end_time"],
    )

In [4]:
dfs = []
missing_signals = []
for signal in cfg['signal'].items():
    print(signal[0])
    print(signal[1])
    try:
        df = extract_signal(
            shot_number=shot_number,
            directory=Path(cfg["raw_data_dir"]),
            signal=signal[0], 
            start_time=start_time,
            end_time=end_time,
        )
        df.columns = [
            f"{signal[1]['abbr']}_{col}" if col != "time" else col
            for col in range(len(df.columns))
        ]

        # # Add a log to validate our assumption about the config and the transform key:
        # logger.debug(f"Signal config for {signal['name']}: {signal}")

        # Add a column to the dataframe for this signal indicating if a transform is present.
        # We'll use the signal's abbreviation to name the column, e.g., 'IP_transform'
        df = align_signal(
            df=df,
            start_time=start_time,
            end_time=end_time,
            fs=cfg["fs_khz"],
        )
        dfs.append(df)
    except Exception as e:
        for channel in range(int(signal[1]['expected_channels'])):
            missing_signals.append((signal[1]['abbr'], channel))

magnetics_high_resolution
{'abbr': 'mhr', 'make_stft': True, 'expected_channels': 8}
ece_cali
{'abbr': 'ece', 'make_stft': True, 'expected_channels': 48}
co2_density
{'abbr': 'co2', 'make_stft': True, 'expected_channels': 4}
gas
{'abbr': 'gas', 'make_stft': False, 'expected_channels': 5}
ech
{'abbr': 'ech', 'make_stft': False, 'expected_channels': 11}
p_inj
{'abbr': 'pin', 'make_stft': False, 'expected_channels': 8}
t_inj
{'abbr': 'tin', 'make_stft': False, 'expected_channels': 8}


In [ ]:
df = pd.concat(dfs, axis=1, join='inner')
for signal_abbr, channel in missing_signals:
    df[f"{signal_abbr}_{channel}"] = np.nan
    df[f"{signal_abbr}_{channel}_state"] = False

In [12]:
samples = split_samples(
    df=df,
    shot_number=shot_number,
    window_ms=cfg["window_ms"],
    hop_ms=cfg["hop_ms"],
    fs_khz=cfg["fs_khz"],
)

In [13]:
samples = remove_empty_samples(samples)

In [ ]:
for sample in samples:
    transformed_samples = {}
    for key, value in sample.items():
        for signal in cfg['signal'].items():
            abbr = signal[1]['abbr']
            cols = [col for col in value.columns if abbr in col]
            transformed_samples[abbr] = identity_transform(
                x=value[cols].to_numpy().T)
            print(abbr, transformed_samples[abbr].shape)
        save_sample(transformed_samples, Path('../data'), key)

mhr (8, 1, 653000)
ece (48, 1, 653000)
co2 (4, 1, 653000)
gas (5, 1, 653000)
ech (11, 1, 653000)
pin (8, 1, 653000)
tin (8, 1, 653000)


In [17]:
first_arr = np.array([list(samples[0].values())[0].iloc[:, 0].values])
transform_shape = stft_transform(x=first_arr).shape
print(first_arr.shape, transform_shape)

(1, 653000) (1, 2551, 513)


In [19]:
for sample in samples:
    transformed_samples = {}
    for key, value in sample.items():
        for signal in cfg['signal'].items():
            abbr = signal[1]['abbr']
            cols = [col for col in value.columns if abbr in col]
            if signal[1]['make_stft']:
                transformed_samples[abbr] = stft_transform(
                    x=value[cols].to_numpy().T,
                    n_fft=cfg["stft"]["n_fft"],
                    hop_length=cfg["stft"]["hop_length"],
                )
            else:
                transformed_samples[abbr] = resample_transform(
                    x=value[cols].to_numpy().T,
                    ref_shape=transform_shape,
                )
            print(abbr, transformed_samples[abbr].shape)
        save_sample(transformed_samples, Path('../data'), key)
        break
    break

mhr (8, 2551, 513)
ece (48, 2551, 513)
co2 (4, 2551, 513)
gas (5, 2551, 1)
ech (11, 2551, 1)
pin (8, 2551, 1)
tin (8, 2551, 1)
